# ⚡ Module 2.4: Auto Logging

**Goal**: Use MLFlow's autolog feature to automatically capture parameters, metrics, and models — zero manual logging required.

---

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

mlflow.set_experiment("02_auto_logging")

# Load data
wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42
)
print("✅ Setup complete!")

## 1. Basic Autolog — The Simplest Approach

`mlflow.autolog()` enables automatic logging for **all supported frameworks**. Just call it once and train your model — MLFlow does the rest.

### What gets automatically logged for scikit-learn:
- ✅ All constructor parameters (hyperparameters)
- ✅ Training metrics (accuracy, f1, etc.)
- ✅ The trained model (with signature)
- ✅ Input example
- ✅ Model signature (input/output schema)

In [ ]:
# Enable autolog for all frameworks
mlflow.autolog()

# Now just train your model — MLFlow handles everything!
with mlflow.start_run(run_name="autolog_random_forest"):
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    
    # That's it! MLFlow automatically logged:
    # - All params: n_estimators, max_depth, min_samples_split, etc.
    # - Training metrics: training_accuracy, training_f1, etc.
    # - The model itself
    # - Model signature and input example
    
    run_id = mlflow.active_run().info.run_id
    print(f"✅ Autolog captured everything!")
    print(f"   Run ID: {run_id}")
    print(f"   Check the MLFlow UI to see all the auto-logged data.")

### Let's see what was automatically logged

In [ ]:
# Fetch the run data to see what autolog captured
run = mlflow.get_run(run_id)

print("📋 Auto-logged PARAMETERS:")
print("=" * 50)
for key, value in sorted(run.data.params.items()):
    print(f"   {key}: {value}")

print(f"\n📊 Auto-logged METRICS:")
print("=" * 50)
for key, value in sorted(run.data.metrics.items()):
    print(f"   {key}: {value:.4f}")

print(f"\n🏷️  Auto-logged TAGS:")
print("=" * 50)
for key, value in sorted(run.data.tags.items()):
    if not key.startswith("mlflow."):
        print(f"   {key}: {value}")

## 2. Framework-Specific Autolog

Instead of `mlflow.autolog()` (which enables for ALL frameworks), you can use framework-specific autolog for more control.

In [ ]:
# Disable global autolog first
mlflow.autolog(disable=True)

# Enable sklearn-specific autolog with custom settings
mlflow.sklearn.autolog(
    log_input_examples=True,      # Log sample input data
    log_model_signatures=True,    # Log input/output schema
    log_models=True,              # Log the trained model
    log_post_training_metrics=True,  # Log metrics after .score() calls
)

with mlflow.start_run(run_name="sklearn_autolog_custom"):
    model = GradientBoostingClassifier(
        n_estimators=200, 
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
    model.fit(X_train, y_train)
    
    # post-training metrics: autolog also captures .score() calls!
    test_score = model.score(X_test, y_test)
    
    print(f"✅ sklearn autolog with custom settings!")
    print(f"   Test score: {test_score:.4f}")

## 3. Autolog with Pipelines

Autolog works with sklearn **Pipelines** too — it captures parameters from all steps.

In [ ]:
with mlflow.start_run(run_name="autolog_pipeline"):
    
    # Create a pipeline with preprocessing and model
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000, C=0.5, random_state=42))
    ])
    
    pipeline.fit(X_train, y_train)
    
    # MLFlow autolog captures:
    # - Scaler params (with_mean, with_std, etc.)
    # - Classifier params (C, max_iter, solver, etc.)
    # - The full pipeline as a model
    
    score = pipeline.score(X_test, y_test)
    print(f"✅ Pipeline autologged!")
    print(f"   Test accuracy: {score:.4f}")
    print(f"   All pipeline step params were captured automatically.")

## 4. Autolog + Manual Logging (Hybrid)

You can **combine** autolog with manual logging to add extra context that autolog doesn't capture.

In [ ]:
with mlflow.start_run(run_name="hybrid_logging"):
    
    # Autolog handles the sklearn stuff
    model = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # MANUAL: Add custom tags that autolog doesn't set
    mlflow.set_tags({
        "author": "sujat",
        "purpose": "model_comparison",
        "dataset_version": "v1.0",
        "notes": "Testing hybrid logging approach"
    })
    
    # MANUAL: Add custom metrics
    mlflow.log_metric("test_accuracy_manual", accuracy_score(y_test, y_pred))
    mlflow.log_metric("num_test_samples", len(y_test))
    
    # MANUAL: Log custom artifacts
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=wine.target_names, ax=ax, cmap='Blues'
    )
    ax.set_title('Confusion Matrix')
    plt.tight_layout()
    fig.savefig("confusion_matrix.png", dpi=150)
    mlflow.log_artifact("confusion_matrix.png", artifact_path="plots")
    plt.show()
    
    import os
    os.remove("confusion_matrix.png")
    
    print("\n✅ Hybrid logging: autolog + custom tags, metrics, and plots!")

## 5. Comparing Autolog vs Manual Logging

In [ ]:
# Disable autolog for a fair comparison
mlflow.sklearn.autolog(disable=True)

# --- MANUAL LOGGING RUN ---
with mlflow.start_run(run_name="manual_logging_example"):
    params = {"n_estimators": 100, "max_depth": 5, "random_state": 42}
    mlflow.log_params(params)
    
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    
    accuracy = accuracy_score(y_test, model.predict(X_test))
    mlflow.log_metric("accuracy", accuracy)
    mlflow.sklearn.log_model(model, artifact_path="model")
    mlflow.set_tag("logging_method", "manual")
    
    manual_run_id = mlflow.active_run().info.run_id
    print(f"Manual: accuracy={accuracy:.4f}")

# --- AUTOLOG RUN ---
mlflow.sklearn.autolog()  # Re-enable

with mlflow.start_run(run_name="autolog_example"):
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    
    accuracy = model.score(X_test, y_test)
    mlflow.set_tag("logging_method", "autolog")
    
    autolog_run_id = mlflow.active_run().info.run_id
    print(f"Autolog: accuracy={accuracy:.4f}")

# Compare what was captured
manual_run = mlflow.get_run(manual_run_id)
autolog_run = mlflow.get_run(autolog_run_id)

print(f"\n📊 Comparison:")
print(f"   Manual - Params logged: {len(manual_run.data.params)}")
print(f"   Autolog - Params logged: {len(autolog_run.data.params)}")
print(f"   Manual - Metrics logged: {len(manual_run.data.metrics)}")
print(f"   Autolog - Metrics logged: {len(autolog_run.data.metrics)}")
print(f"\n💡 Autolog captures MORE data with LESS code!")

## 🔑 Key Takeaways

| Approach | Pros | Cons | Best For |
|----------|------|------|----------|
| `mlflow.autolog()` | Zero effort, comprehensive | Less control, captures everything | Quick experiments |
| `mlflow.sklearn.autolog()` | Framework-specific control | Still automatic | Production sklearn projects |
| Manual logging | Full control | More code to write | Custom workflows |
| **Hybrid** | Best of both worlds | Slight complexity | **Recommended for most projects** |

### Recommendation
> 🏆 Use the **hybrid approach** in practice: enable autolog for automatic capture, then add custom tags, metrics, and artifacts manually.

---

## ➡️ Next: `05_nested_runs.ipynb` — Organize complex experiments with parent-child runs